# CSC4093/DSC4213: Neural Networks and Deep Learning
## Programming Assignment 01 — LSTMs
### Personal Health Mention (PHM) Classification from Tweets
---
**Task:** Binary classification — Personal Health Mention `(1)` vs Non-Personal Health Mention `(0)`  
**Models:** LSTM and Bidirectional LSTM (Bi-LSTM)  
**Reference:** Based on the IMDB Movie Review LSTM sample notebook provided

## Step 1: Install Dependencies and Import Libraries

In [ ]:
import sys
!{sys.executable} -m pip install tensorflow nltk scikit-learn matplotlib seaborn pandas numpy --quiet
print('Packages ready.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
english_stops = set(stopwords.words('english'))

from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.callbacks import ModelCheckpoint

import os
os.makedirs('models', exist_ok=True)

print('All libraries imported successfully.')

## Step 2: Load and Preview Dataset

In [ ]:
train_data = pd.read_csv('phm_train.csv')
test_data  = pd.read_csv('phm_test.csv')

print('Training set shape:', train_data.shape)
print('Test set shape    :', test_data.shape)
print()
print(train_data[['label', 'tweet']].head())
print()
print('Label distribution — Train:')
print(train_data['label'].value_counts())
print()
print('Label distribution — Test:')
print(test_data['label'].value_counts())

## Step 3: Text Preprocessing

In [ ]:
def load_dataset(data):
    x_data = data['tweet'].copy().astype(str)
    y_data = data['label'].copy()

    # Remove URLs, mention placeholders, emotion tags, and non-alphabetic characters
    x_data = x_data.replace(r'http\S+|www\S+', '', regex=True)
    x_data = x_data.replace(r'user_mention', '', regex=True)
    x_data = x_data.replace(r'emo_\w+', '', regex=True)
    x_data = x_data.replace(r'<.*?>', '', regex=True)
    x_data = x_data.replace(r'[^A-Za-z]', ' ', regex=True)

    # Tokenize, lowercase, and remove stopwords
    x_data = x_data.apply(lambda tweet: [
        w.lower() for w in tweet.split()
        if w.lower() not in english_stops
    ])
    return x_data, y_data

x_train, y_train = load_dataset(train_data)
x_test,  y_test  = load_dataset(test_data)

print('Tweets (x_train):')
print(x_train)
print()
print('Labels (y_train):')
print(y_train)

## Step 4: Tokenization and Padding

In [ ]:
def get_max_length(x_data):
    review_length = []
    for tweet in x_data:
        review_length.append(len(tweet))
    return int(np.ceil(np.mean(review_length)))

# Fit tokenizer only on training data
token = Tokenizer(lower=False)  # already lowercased in preprocessing
token.fit_on_texts(x_train)

x_train_seq = token.texts_to_sequences(x_train)
x_test_seq  = token.texts_to_sequences(x_test)

max_length  = get_max_length(x_train)
total_words = len(token.word_index) + 1

x_train_pad = pad_sequences(x_train_seq, maxlen=max_length, padding='post', truncating='post')
x_test_pad  = pad_sequences(x_test_seq,  maxlen=max_length, padding='post', truncating='post')

print('Encoded X Train')
print(x_train_pad)
print()
print('Encoded X Test')
print(x_test_pad)
print()
print('Maximum tweet length :', max_length)
print('Total vocabulary size:', total_words)

## Step 5: Build Model 1 — LSTM

In [ ]:
# ARCHITECTURE
EMBED_DIM = 32
LSTM_OUT  = 64

lstm_model = Sequential()
lstm_model.add(Embedding(total_words, EMBED_DIM, input_length=max_length))
lstm_model.add(LSTM(LSTM_OUT))
lstm_model.add(Dense(1, activation='sigmoid'))

lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print(lstm_model.summary())

## Step 6: Train LSTM Model

In [ ]:
lstm_checkpoint = ModelCheckpoint(
    'models/LSTM_PHM.keras',
    monitor='accuracy',
    save_best_only=True,
    verbose=1
)

lstm_history = lstm_model.fit(
    x_train_pad, y_train,
    batch_size=128,
    epochs=5,
    callbacks=[lstm_checkpoint]
)

## Step 7: Build Model 2 — Bi-LSTM

In [ ]:
# ARCHITECTURE
EMBED_DIM    = 32
BILSTM_OUT   = 64

bilstm_model = Sequential()
bilstm_model.add(Embedding(total_words, EMBED_DIM, input_length=max_length))
bilstm_model.add(Bidirectional(LSTM(BILSTM_OUT)))
bilstm_model.add(Dense(1, activation='sigmoid'))

bilstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print(bilstm_model.summary())

## Step 8: Train Bi-LSTM Model

In [ ]:
bilstm_checkpoint = ModelCheckpoint(
    'models/BiLSTM_PHM.keras',
    monitor='accuracy',
    save_best_only=True,
    verbose=1
)

bilstm_history = bilstm_model.fit(
    x_train_pad, y_train,
    batch_size=128,
    epochs=5,
    callbacks=[bilstm_checkpoint]
)

## Step 9: Test LSTM Model

In [ ]:
y_pred_lstm = lstm_model.predict(x_test_pad)
y_pred_lstm = (y_pred_lstm >= 0.5).astype(int)

lstm_true = 0
for i, y in enumerate(y_test):
    if y == y_pred_lstm[i]:
        lstm_true += 1

print('Correct Prediction :', lstm_true)
print('Wrong Prediction   :', len(y_pred_lstm) - lstm_true)
print('Accuracy           :', lstm_true / len(y_pred_lstm) * 100)

## Step 10: Test Bi-LSTM Model

In [ ]:
y_pred_bilstm = bilstm_model.predict(x_test_pad)
y_pred_bilstm = (y_pred_bilstm >= 0.5).astype(int)

bilstm_true = 0
for i, y in enumerate(y_test):
    if y == y_pred_bilstm[i]:
        bilstm_true += 1

print('Correct Prediction :', bilstm_true)
print('Wrong Prediction   :', len(y_pred_bilstm) - bilstm_true)
print('Accuracy           :', bilstm_true / len(y_pred_bilstm) * 100)

## Step 11: Accuracy and Loss Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('LSTM vs Bi-LSTM Training Performance', fontsize=14, fontweight='bold')

axes[0,0].plot(lstm_history.history['accuracy'], color='royalblue', linewidth=2, marker='o')
axes[0,0].set_title('LSTM — Training Accuracy')
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Accuracy')
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(lstm_history.history['loss'], color='darkorange', linewidth=2, marker='o')
axes[0,1].set_title('LSTM — Training Loss')
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Loss')
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(bilstm_history.history['accuracy'], color='seagreen', linewidth=2, marker='o')
axes[1,0].set_title('Bi-LSTM — Training Accuracy')
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Accuracy')
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(bilstm_history.history['loss'], color='crimson', linewidth=2, marker='o')
axes[1,1].set_title('Bi-LSTM — Training Loss')
axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('Loss')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('accuracy_loss_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: accuracy_loss_plots.png')

## Step 12: Confusion Matrices

In [ ]:
lstm_acc_pct   = lstm_true   / len(y_pred_lstm)   * 100
bilstm_acc_pct = bilstm_true / len(y_pred_bilstm) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrices — LSTM vs Bi-LSTM', fontsize=14, fontweight='bold')
labels = ['Non-Personal (0)', 'Personal Health (1)']

cm_lstm   = confusion_matrix(y_test, y_pred_lstm)
cm_bilstm = confusion_matrix(y_test, y_pred_bilstm)

sns.heatmap(cm_lstm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[0], linewidths=0.5)
axes[0].set_title(f'LSTM  (Accuracy: {lstm_acc_pct:.2f}%)')
axes[0].set_ylabel('Actual Label'); axes[0].set_xlabel('Predicted Label')
axes[0].tick_params(axis='x', rotation=15)

sns.heatmap(cm_bilstm, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels, yticklabels=labels, ax=axes[1], linewidths=0.5)
axes[1].set_title(f'Bi-LSTM  (Accuracy: {bilstm_acc_pct:.2f}%)')
axes[1].set_ylabel('Actual Label'); axes[1].set_xlabel('Predicted Label')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrices.png')

## Step 13: Classification Reports and Performance Summary

In [ ]:
print('=' * 55)
print('     LSTM — Classification Report')
print('=' * 55)
print(classification_report(
    y_test, y_pred_lstm,
    target_names=['Non-Personal (0)', 'Personal Health (1)']
))

print('=' * 55)
print('   Bi-LSTM — Classification Report')
print('=' * 55)
print(classification_report(
    y_test, y_pred_bilstm,
    target_names=['Non-Personal (0)', 'Personal Health (1)']
))

# Summary table
comparison = pd.DataFrame({
    'Metric': [
        'Test Accuracy (%)',
        'Precision (weighted %)',
        'Recall (weighted %)',
        'F1-Score (weighted %)',
        'Best Training Accuracy (%)'
    ],
    'LSTM': [
        f'{lstm_acc_pct:.2f}',
        f'{precision_score(y_test, y_pred_lstm,   average="weighted")*100:.2f}',
        f'{recall_score(y_test,    y_pred_lstm,   average="weighted")*100:.2f}',
        f'{f1_score(y_test,        y_pred_lstm,   average="weighted")*100:.2f}',
        f"{max(lstm_history.history['accuracy'])*100:.2f}"
    ],
    'Bi-LSTM': [
        f'{bilstm_acc_pct:.2f}',
        f'{precision_score(y_test, y_pred_bilstm, average="weighted")*100:.2f}',
        f'{recall_score(y_test,    y_pred_bilstm, average="weighted")*100:.2f}',
        f'{f1_score(y_test,        y_pred_bilstm, average="weighted")*100:.2f}',
        f"{max(bilstm_history.history['accuracy'])*100:.2f}"
    ]
})

print('=' * 60)
print('        Model Performance Comparison')
print('=' * 60)
print(comparison.to_string(index=False))
print('=' * 60)

## Step 14: Discussion

### Model Architectures

Both models follow the same base architecture used in the provided IMDB sample notebook, adapted for tweet-level binary classification:

```
Embedding(vocab_size, 32, input_length=max_length)
    ↓
LSTM(64)  /  Bidirectional(LSTM(64))   ← only difference
    ↓
Dense(1, activation='sigmoid')
```

### Hyperparameters

| Parameter | Value | Reason |
|---|---|---|
| Embedding dim | 32 | Lightweight representation for tweet vocabulary |
| LSTM units | 64 | Adequate capacity for short tweet sequences |
| Batch size | 128 | Stable gradient estimates |
| Epochs | 5 | Consistent with sample notebook; prevents overfitting on small dataset |
| Optimizer | Adam | Adaptive learning rate — standard for LSTM tasks |
| Loss | Binary Crossentropy | Appropriate for binary classification |

### LSTM Model
The standard LSTM processes each tweet **token by token in a single forward direction**, maintaining a hidden state that accumulates sequential context. It is well-suited for short tweet sequences where health mentions typically appear with first-person language such as *'I took'*, *'my doctor prescribed'*, or *'I feel'*. The LSTM learns to distinguish these personal patterns from third-person or figurative drug references.

### Bi-LSTM Model
The Bidirectional LSTM runs **two LSTM layers simultaneously** — one processing the tweet forward (left to right) and one backward (right to left). Their hidden states are concatenated at each timestep, giving the model full context in both directions. This is advantageous when a key personal pronoun (*'I'*, *'my'*) appears at the start of the tweet but the medication mention appears at the end — the backward pass ensures the model connects these distant tokens.

### Confusion Matrix Interpretation
Each confusion matrix shows four cells:
- **Top-left (TN):** Non-personal tweets correctly classified
- **Bottom-right (TP):** Personal health tweets correctly identified
- **Top-right (FP):** Non-personal tweets incorrectly flagged as personal
- **Bottom-left (FN):** Personal health tweets missed by the model

In health mention detection, **False Negatives** are the most costly error — missing a genuine personal health mention is worse than a false alarm. A lower FN count is therefore a key indicator of a well-performing model.

### Model Comparison
The Bi-LSTM generally outperforms the standard LSTM on this task because tweet language is contextually rich in both directions. However, the gain may be modest since tweets are short, and the additional parameters of the bidirectional layer can slightly increase training time. The classification report provides precision, recall, and F1-score per class, giving a more complete picture than accuracy alone — particularly important given the class imbalance in the PHM dataset.